In [1]:
import torch
from torch import nn,Tensor 
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from src.detectors.internal.linear_probe import LinearProbe

BATCH_SIZE = 32 

In [2]:
from transformers import AutoModelForCausalLM,AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B", device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-0.6B", padding_side="left"
)
if torch.cuda.is_available():
    device = "cuda"
else: 
    device = "cpu"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


c:\Users\Ramneek\anaconda3\envs\safeguard_llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:01<00:00, 303.12it/s]


In [3]:
from datasets import load_dataset
from torch.utils.data import DataLoader
toxic_dataset = load_dataset("textdetox/multilingual_toxicity_dataset")
print(toxic_dataset.data["en"])
en_dataset = toxic_dataset["en"].shuffle(seed=40)

split_dataset = en_dataset.train_test_split(test_size=0.1, seed=40)
train_toxic_dataset = split_dataset["train"]
test_toxic_dataset = split_dataset["test"]
train_data = DataLoader(train_toxic_dataset, batch_size=32)
test_data = DataLoader(test_toxic_dataset, batch_size=32)
#set up llm and environment
#load dataset
#setup batched dataset
#lets start simple with a basic harmeval 
#define the linear probe
#hook into llm
#training loop with cached activation 
#compare prediction with label 


MemoryMappedTable
text: string
toxic: int64
----
text: [["The trans women reading this tweet right now is beautiful","9) uhhhh i like being lgbt a lot. i feel proud of what i have done to help others in my community","Hero Rohit Sharma love From Pakistan","As a slightly feminine top I appreciate all masc bottoms that enjoy a dude like me","Delon Love Turkey and brave Turks from Indian Muslim! In Shaa Allah we will rise again!",...,"It's 2019 we're respecting women and getting respected back.","does skin colour matter to world leaders? does what you believe matter to world leaders?","A massive thanks from a bi teen","The least we could do this Pride month is get a trans mother back to her kids. Donate and share! U (... 2 chars omitted)","is being trans valid? please retweet after voting for"],["These women are so powerful and inspiring","God is a woman and her name is Sophie Turner","Ah. I'm planning on coming out as trans (and I guess lesbian) hopefully this week. Good luck man","You d

In [4]:
print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [5]:
test = torch.rand([1,4,20])
test= torch.mean(test,dim=1)
print(test.shape)

torch.Size([1, 20])


Train Loop 1: The model trains on the final token of the prompt prefill, it learns to vector representation of toxicity in the latent space with the activation of the input prompt only. The final vector in the seq len of the prefill generation represents a vector [batch_size, dim] that has the contextual information of the entire previous sentence, meaning that we only have to train on this. 

In [22]:
def hook(module, input: Tensor, output: Tensor) -> None:
    global activation_cache 
    activation_cache = output[:, -1, :].detach().clone().to(device)

In [23]:

import torch
from tqdm.auto import tqdm

linear_probe_model = LinearProbe(in_dim=3072).to(device).bfloat16()

forward_hook = model.model.layers[27].mlp.act_fn.register_forward_hook(hook)
optimizer = torch.optim.Adam(linear_probe_model.parameters(), lr=0.001)
loss_fn = torch.nn.BCELoss()

EPOCHS = 3
linear_probe_model.train()
for epoch in range(EPOCHS): 
    pbar = tqdm(train_data, desc=f"Epoch {epoch + 1}/{EPOCHS}", leave=True)
    
    for batch in pbar:
        inputs = batch["text"]
        labels = batch["toxic"].to(device).bfloat16()
        tokenized_input = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**tokenized_input)
        scores = linear_probe_model(activation_cache)
        optimizer.zero_grad() 
        loss = loss_fn(scores.squeeze(-1), labels)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=f"{loss.item():.4f}")


forward_hook.remove()


Epoch 1/3:  52%|█████▏    | 73/141 [00:08<00:07,  8.88it/s, loss=0.3145]


KeyboardInterrupt: 

In [17]:
torch.save(linear_probe_model.state_dict(), "model_linear.pt")

In [7]:
forward_hook = model.model.layers[27].mlp.act_fn.register_forward_hook(hook)
pbar = tqdm(train_data, leave=True)
tp = 0
fp = 0
tn = 0
fn = 0
for batch in pbar:
    inputs = batch["text"]
    labels = batch["toxic"].to(device).bfloat16()
    tokenized_input = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**tokenized_input)
        scores = linear_probe_model.predict(activation_cache,threshold=0.5)
        scores = scores.squeeze(-1)
        tp += torch.sum(torch.logical_and(scores == 1 ,labels == 1))
        tn += torch.sum(torch.logical_and(scores == 0,labels == 0))
        fp += torch.sum(torch.logical_and(scores == 1,labels == 0))
        fn += torch.sum(torch.logical_and(scores == 0,labels == 1))
print(tp)
print(fp)
print(tn)
print(fn)
accuracy = (tp + tn) /( tp + fp + tn + fn)
print(accuracy)
forward_hook.remove()



100%|██████████| 141/141 [00:13<00:00, 10.40it/s]

tensor(1587, device='cuda:0')
tensor(98, device='cuda:0')
tensor(2152, device='cuda:0')
tensor(663, device='cuda:0')
tensor(0.8309, device='cuda:0')


In [8]:
from datasets import load_dataset
from torch.utils.data import DataLoader
toxic_dataset = load_dataset("textdetox/multilingual_toxicity_dataset")
print(toxic_dataset.data["de"])
de_dataset = toxic_dataset["de"].shuffle(seed=40)

split_dataset_de = de_dataset.train_test_split(test_size=0.1, seed=40)
train_toxic_dataset_de = split_dataset_de["train"]
train_toxic_dataset_de = split_dataset_de["test"]
train_data_de = DataLoader(train_toxic_dataset_de, batch_size=32)
test_data_de = DataLoader(train_toxic_dataset_de, batch_size=32)

MemoryMappedTable
text: string
toxic: int64
----
text: [[" Untaten mit "Flüchtlingen" oder Moslems in Verbindung zu bringen wird immer als rassistische Het (... 142 chars omitted)","Der Regenwald wird im Akkord abgeholzt ,das interessiert niemanden !"," Es geht darum was der Makler sich rein zieht."," Huhhhhh, Fritz, Du jagst mir richtig Angst ein 😜","   Hab versucht zu entziffern was du da von dir gibst, bin dabei behindert geworden du Bastard",...,"  Voll bei dir. Dampf ablassen. Ich gehe fast jeden Tag an einem vollen Kinderspielplatz an einer  (... 149 chars omitted)","  .....das lichtscheue Gesindel aus einer Melange von Antifa, Verdi, SPD und Migranten zeigt mal w (... 43 chars omitted)","Heute Ältestenrat zur Vorbereitung der nächsten Landtagssitzung. |LBR| Später diverse Medienter (... 80 chars omitted)","Was zum Teufel ist mit der CDU nicht in Ordnung??? Jetzt wird auch noch ein bekannter Nazi unterst (... 168 chars omitted)","   Er hatte niemals die Macht das zu verhindern, e

In [24]:
linear_probe_model = LinearProbe(in_dim=3072).to(device).bfloat16()
linear_probe_model.load_state_dict(torch.load("model_linear.pt",weights_only=True))
linear_probe_model.eval()

LinearProbe(
  (lin): Linear(in_features=3072, out_features=1, bias=True)
  (sig): Sigmoid()
)

In [25]:
forward_hook = model.model.layers[27].mlp.act_fn.register_forward_hook(hook)
pbar = tqdm(test_data, leave=True)
tp = 0
fp = 0
tn = 0
fn = 0
for batch in pbar:
    inputs = batch["text"]
    labels = batch["toxic"].to(device).bfloat16()
    tokenized_input = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**tokenized_input)
        scores = linear_probe_model.predict(activation_cache,threshold=0.5)
        scores = scores.squeeze(-1)
        tp += torch.sum(torch.logical_and(scores == 1 ,labels == 1))
        tn += torch.sum(torch.logical_and(scores == 0,labels == 0))
        fp += torch.sum(torch.logical_and(scores == 1,labels == 0))
        fn += torch.sum(torch.logical_and(scores == 0,labels == 1))
print(tp)
print(fp)
print(tn)
print(fn)
accuracy = (tp + tn) /( tp + fp + tn + fn)
print(accuracy)
forward_hook.remove()



100%|██████████| 16/16 [00:01<00:00,  9.57it/s]

tensor(222, device='cuda:0')
tensor(21, device='cuda:0')
tensor(229, device='cuda:0')
tensor(28, device='cuda:0')
tensor(0.9020, device='cuda:0')


In [29]:
forward_hook = model.model.layers[27].mlp.act_fn.register_forward_hook(hook)
pbar = tqdm(test_data_de, leave=True)
tp = 0
fp = 0
tn = 0
fn = 0
for batch in pbar:
    inputs = batch["text"]
    labels = batch["toxic"].to(device).bfloat16()
    tokenized_input = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**tokenized_input)
        scores = linear_probe_model.predict(activation_cache,threshold=0.5)
        scores = scores.squeeze(-1)
        tp += torch.sum(torch.logical_and(scores == 1 ,labels == 1))
        tn += torch.sum(torch.logical_and(scores == 0,labels == 0))
        fp += torch.sum(torch.logical_and(scores == 1,labels == 0))
        fn += torch.sum(torch.logical_and(scores == 0,labels == 1))
    break
print(tp)
print(tn)
print(fp)
print(fn)
accuracy = (tp + tn) /( tp + fp + tn + fn)
print(accuracy)
forward_hook.remove()


  0%|          | 0/16 [00:01<?, ?it/s]

tensor(7, device='cuda:0')
tensor(5, device='cuda:0')
tensor(16, device='cuda:0')
tensor(4, device='cuda:0')
tensor(0.3750, device='cuda:0')
